# ARIMA forecasting experiment

All diagnostics and model selection use only the configured training/validation history. The test partition is evaluated once after the order is frozen. ARIMA parameters are refit on the configured schedule; between refits, the fitted state is updated with observations available at each origin. Horizons count subsequent observed TRM records.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
from trm_forecast.data.audit import audit_trm
from trm_forecast.evaluation.arima import evaluate_arima
from trm_forecast.evaluation.persistence import evaluate_persistence
from trm_forecast.evaluation.protocol import PartitionSpec, TemporalEvaluationProtocol
from trm_forecast.models.arima import stationarity_diagnostics

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
config = yaml.safe_load((ROOT / 'configs/base.yaml').read_text())
data_path = ROOT / 'data/interim/trm_normalized.csv'
frame = pd.read_csv(data_path, parse_dates=['date'])
series = frame.set_index('date')['trm'].sort_index()
evaluation = config['evaluation']
specs = tuple(PartitionSpec(name, values['start'], values['end']) for name, values in evaluation['partitions'].items())
protocol = TemporalEvaluationProtocol(evaluation['as_of_date'], tuple(evaluation['horizons']), specs)
result = protocol.split(series)
result.summaries

## Train-only stationarity diagnostics

ADF has a unit-root null while KPSS has a stationarity null; disagreement is possible and neither test establishes the best forecasting specification.

In [ ]:
train = result.partitions['train']
diagnostics = stationarity_diagnostics(train)
diagnostics

## Validation selection

Each candidate is evaluated only on validation. We select the lowest mean validation MAE across horizons, then freeze that order before test evaluation.

In [ ]:
horizons = tuple(evaluation['horizons'])
candidate_rows = []
for order in config['arima']['candidate_orders']:
    _, candidate_metrics = evaluate_arima(result, tuple(order), horizons, ('validation',), config['arima']['min_history'], config['arima']['maxiter'])
    candidate_rows.append({'order': tuple(order), 'mean_validation_mae': candidate_metrics['mae'].mean(), 'failed_predictions': int(candidate_metrics['failed_predictions'].sum())})
selection = pd.DataFrame(candidate_rows).sort_values('mean_validation_mae')
selection
selected_order = selection.iloc[0]['order']
selected_order

## Frozen comparison

Persistence and ARIMA use the same eligible origins and targets. Failed ARIMA fits remain in the prediction table and are counted explicitly; metrics use successful forecasts only.

In [ ]:
persistence_predictions, persistence_metrics = evaluate_persistence(result, horizons)
arima_predictions, arima_metrics = evaluate_arima(result, tuple(selected_order), horizons, ('validation', 'test'), config['arima']['min_history'], config['arima']['maxiter'])
comparison = pd.concat([persistence_metrics.assign(model='persistence'), arima_metrics.assign(model='arima')], ignore_index=True)
comparison.sort_values(['partition', 'horizon', 'model'])

In [ ]:
arima_predictions.groupby(['partition', 'status']).size().rename('predictions').reset_index()